# (노트) CAM - cat, dog 

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- categories: [빅데이터분석]

### import

In [1]:
from fastai.vision.all import * 
import torch 

### data 

In [9]:
path= untar_data(URLs.PETS)/'images'

In [10]:
files=get_image_files(path) 

In [11]:
files[2] # txt 파일의 3번째 목록

Path('/home/cgb3/.fastai/data/oxford-iiit-pet/images/japanese_chin_50.jpg')

In [12]:
def label_func(f):
    if f[0].isupper():
        return 'cat' 
    else: 
        return 'dog' 

In [13]:
dls=ImageDataLoaders.from_name_func(path,files,label_func,item_tfms=Resize(512),valid_pct=0.2,seed=21) 

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call,so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.

### learn 

In [ ]:
lrnr=cnn_learner(dls,resnet18,metrics=error_rate)
lrnr.fine_tune(10)

### 모형뜯어보기

`-` 샘플로 하나의 관측치를 만든다. 

In [ ]:
img=PILImage.create(get_image_files(path)[10])
img

`-` 전체네트워크를 1,2로 나눈다. 

In [ ]:
net1=lrnr.model[0]
net2=lrnr.model[1]

In [ ]:
net2(net1(x)), lrnr.model(x)

`-` 네트워크1의 출력결과

In [ ]:
net1(x).shape

In [ ]:
lrnr.model[0](x)[0].shape

In [ ]:
lrnr.model[1][-1].weight

In [ ]:
torch.einsum('ck,kij->cij',net2[8].weight,net1(x)[0]).shape

In [ ]:
cam_map[0]

In [ ]:
cam_map[1].to("cpu").detach().sum()

In [ ]:
lrnr.predict(PILImage.create('2021-08-25-Hani01.jpeg'))

In [ ]:
lrnr.model(x)

In [ ]:
cam_map[1].sum()

### CAM one_batch

`-` 개 혹은 늑대라고 판단한 근거를 알아보자. 

#### 하니

In [ ]:
x, =first(dls.test_dl([PILImage.create('2021-08-25-Hani01.jpeg')]))
dls.vocab

In [ ]:
lrnr.model(x)

In [ ]:
category=torch.argmax(lrnr.model(x)).to("cpu").tolist()

In [ ]:
category

In [ ]:
cam_map= torch.einsum('ck,kij->cij',lrnr.model[1][-1].weight,lrnr.model[0](x)[0])
x_dec=TensorImage(dls.train.decode((x,))[0][0])

fig,((ax1,ax2),(ax3,ax4))= plt.subplots(2,2)
x_dec.show(ctx=ax1)

x_dec.show(ctx=ax2)
ax2.imshow(cam_map[category].detach().cpu(),alpha=0.8,extent=(0,512,512,0),interpolation='bilinear',cmap='magma');
ax3.imshow(cam_map[category].detach().cpu(),alpha=0.8,extent=(0,512,512,0),cmap='magma');
ax4.imshow(cam_map[category].detach().cpu(),alpha=0.8,extent=(0,512,512,0),interpolation='bilinear',cmap='magma');

fig.set_figwidth(12)
fig.set_figheight(12)
fig.tight_layout()

#### 다른샘플들

In [ ]:
img=PILImage.create(get_image_files(path)[10])
x,  = first(dls.test_dl([img]))
category=torch.argmax(lrnr.model(x)).to("cpu").tolist()
cam_map= torch.einsum('ck,kij->cij',lrnr.model[1][-1].weight,lrnr.model[0](x)[0])
x_dec=TensorImage(dls.train.decode((x,))[0][0])
fig,((ax1,ax2),(ax3,ax4))= plt.subplots(2,2)
x_dec.show(ctx=ax1)
x_dec.show(ctx=ax2)
ax2.imshow(cam_map[category].detach().cpu(),alpha=0.8,extent=(0,512,512,0),interpolation='bilinear',cmap='magma');
ax3.imshow(cam_map[category].detach().cpu(),alpha=0.8,extent=(0,512,512,0),cmap='magma');
ax4.imshow(cam_map[category].detach().cpu(),alpha=0.8,extent=(0,512,512,0),interpolation='bilinear',cmap='magma');
fig.set_figwidth(12)
fig.set_figheight(12)
fig.tight_layout()

### CAM 결과 확인

In [ ]:
fig, ax = plt.subplots(5,5)
k=0
for i in range(5):
    for j in range(5): 
        img=PILImage.create(get_image_files(path)[k])
        x, = first(dls.test_dl([img]))
        x_dec=TensorImage(dls.train.decode((x,))[0][0])
        category=torch.argmax(lrnr.model(x)).to("cpu").tolist()
        cam_map= torch.einsum('ck,kij->cij',lrnr.model[1][-1].weight,lrnr.model[0](x)[0])
        x_dec.show(ctx=ax[i][j])
        ax[i][j].imshow(cam_map[category].detach().cpu(),alpha=0.6,extent=(0,512,512,0),interpolation='bilinear',cmap='magma');
        k=k+1
fig.set_figwidth(16)
fig.set_figheight(16)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(5,5)
k=25
for i in range(5):
    for j in range(5): 
        img=PILImage.create(get_image_files(path)[k])
        x, = first(dls.test_dl([img]))
        x_dec=TensorImage(dls.train.decode((x,))[0][0])
        category=torch.argmax(lrnr.model(x)).to("cpu").tolist()
        cam_map= torch.einsum('ck,kij->cij',lrnr.model[1][-1].weight,lrnr.model[0](x)[0])
        x_dec.show(ctx=ax[i][j])
        ax[i][j].imshow(cam_map[category].detach().cpu(),alpha=0.6,extent=(0,512,512,0),interpolation='bilinear',cmap='magma');
        k=k+1
fig.set_figwidth(16)
fig.set_figheight(16)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(5,5)
k=50
for i in range(5):
    for j in range(5): 
        img=PILImage.create(get_image_files(path)[k])
        x, = first(dls.test_dl([img]))
        x_dec=TensorImage(dls.train.decode((x,))[0][0])
        category=torch.argmax(lrnr.model(x)).to("cpu").tolist()
        cam_map= torch.einsum('ck,kij->cij',lrnr.model[1][-1].weight,lrnr.model[0](x)[0])
        x_dec.show(ctx=ax[i][j])
        ax[i][j].imshow(cam_map[category].detach().cpu(),alpha=0.6,extent=(0,512,512,0),interpolation='bilinear',cmap='magma');
        k=k+1
fig.set_figwidth(16)
fig.set_figheight(16)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(5,5)
k=75
for i in range(5):
    for j in range(5): 
        img=PILImage.create(get_image_files(path)[k])
        x, = first(dls.test_dl([img]))
        x_dec=TensorImage(dls.train.decode((x,))[0][0])
        category=torch.argmax(lrnr.model(x)).to("cpu").tolist()
        cam_map= torch.einsum('ck,kij->cij',lrnr.model[1][-1].weight,lrnr.model[0](x)[0])
        x_dec.show(ctx=ax[i][j])
        ax[i][j].imshow(cam_map[category].detach().cpu(),alpha=0.6,extent=(0,512,512,0),interpolation='bilinear',cmap='magma');
        k=k+1
fig.set_figwidth(16)
fig.set_figheight(16)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(5,5)
k=100
for i in range(5):
    for j in range(5): 
        img=PILImage.create(get_image_files(path)[k])
        x, = first(dls.test_dl([img]))
        x_dec=TensorImage(dls.train.decode((x,))[0][0])
        category=torch.argmax(lrnr.model(x)).to("cpu").tolist()
        cam_map= torch.einsum('ck,kij->cij',lrnr.model[1][-1].weight,lrnr.model[0](x)[0])
        x_dec.show(ctx=ax[i][j])
        ax[i][j].imshow(cam_map[category].detach().cpu(),alpha=0.6,extent=(0,512,512,0),interpolation='bilinear',cmap='magma');
        k=k+1
fig.set_figwidth(16)
fig.set_figheight(16)
fig.tight_layout()